In [119]:
import pandas as pd
import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import seaborn as sns

raw_df = pd.read_csv("weatherAUS.csv")

year = pd.to_datetime(raw_df.Date).dt.year

train_df = raw_df[year < 2015] 
val_df = raw_df[year == 2015]
test_df = raw_df[year > 2015]

In [120]:
input_cols = list(raw_df.columns)[1:-1]
target_col = "RainTomorrow"

In [121]:
train_inputs = train_df[input_cols].copy()
train_targets = train_df[target_col].copy()
val_inputs = val_df[input_cols].copy()
val_targets = val_df[target_col].copy()
test_inputs = test_df[input_cols].copy()
test_targets = test_df[target_col].copy()

In [122]:
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes(exclude=np.number).columns.tolist()
print(f"Numeric columns: {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")

Numeric columns: ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm']
Categorical columns: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']


In [123]:
train_inputs[numeric_cols].describe()

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm
count,100282.000000,100484.000000,98987.00000,62532.000000,59010.000000,93395.000000,99564.000000,99556.000000,99428.000000,99514.000000,91218.000000,91252.000000,63913.000000,62888.000000,99913.000000,100040.000000
mean,11.992238,22.981287,2.38158,5.282092,7.593952,40.265892,14.127898,18.777813,68.694231,51.552445,1017.520623,1015.144023,4.316712,4.421559,16.812393,21.506126
std,6.336489,6.994097,8.51861,3.949170,3.790480,13.729898,9.008386,8.882133,18.981489,20.739573,7.075771,6.998990,2.867476,2.693835,6.393443,6.832900
min,-8.500000,-4.100000,0.00000,0.000000,0.000000,6.000000,0.000000,0.000000,0.000000,0.000000,980.500000,979.000000,0.000000,0.000000,-5.900000,-5.100000
25%,7.500000,17.800000,0.00000,2.600000,4.800000,31.000000,7.000000,13.000000,57.000000,37.000000,1012.800000,1010.400000,1.000000,2.000000,12.200000,16.500000
50%,11.800000,22.400000,0.00000,4.600000,8.400000,39.000000,13.000000,19.000000,70.000000,52.000000,1017.500000,1015.100000,5.000000,5.000000,16.500000,20.900000
75%,16.600000,27.900000,0.80000,7.200000,10.600000,48.000000,20.000000,24.000000,83.000000,66.000000,1022.300000,1019.900000,7.000000,7.000000,21.300000,26.100000
max,33.900000,48.100000,371.00000,82.400000,14.300000,135.000000,87.000000,87.000000,100.000000,100.000000,1041.000000,1039.600000,9.000000,9.000000,40.200000,46.100000


In [124]:
train_df[categorical_cols].describe()

,Location,WindGustDir,WindDir9am,WindDir3pm,RainToday
count,101018,93353,93273,98651,98987
unique,49,16,16,16,2
top,Canberra,W,N,SE,No
freq,2529,6841,8306,7731,76701


In [125]:
from sklearn.impute import SimpleImputer

In [126]:
numeric_imputer = SimpleImputer(strategy="mean")
categorical_imputer = SimpleImputer(strategy="most_frequent")

In [127]:
numeric_imputer.fit(train_inputs[numeric_cols])

train_inputs[numeric_cols] = numeric_imputer.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = numeric_imputer.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = numeric_imputer.transform(test_inputs[numeric_cols])

categorical_imputer.fit(train_inputs[categorical_cols])

train_inputs[categorical_cols] = categorical_imputer.transform(train_inputs[categorical_cols])
val_inputs[categorical_cols] = categorical_imputer.transform(val_inputs[categorical_cols])
test_inputs[categorical_cols] = categorical_imputer.transform(test_inputs[categorical_cols])

In [128]:
raw_df[numeric_cols].isna().sum()

MinTemp           1485
MaxTemp           1261
Rainfall          3261
Evaporation      62790
Sunshine         69835
WindGustSpeed    10263
WindSpeed9am      1767
WindSpeed3pm      3062
Humidity9am       2654
Humidity3pm       4507
Pressure9am      15065
Pressure3pm      15028
Cloud9am         55888
Cloud3pm         59358
Temp9am           1767
Temp3pm           3609
dtype: int64

In [129]:
train_inputs[numeric_cols].isna().sum()

MinTemp          0
MaxTemp          0
Rainfall         0
Evaporation      0
Sunshine         0
WindGustSpeed    0
WindSpeed9am     0
WindSpeed3pm     0
Humidity9am      0
Humidity3pm      0
Pressure9am      0
Pressure3pm      0
Cloud9am         0
Cloud3pm         0
Temp9am          0
Temp3pm          0
dtype: int64

In [130]:
raw_df[numeric_cols].describe()

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm
count,143975.000000,144199.000000,142199.000000,82670.000000,75625.000000,135197.000000,143693.000000,142398.000000,142806.000000,140953.000000,130395.00000,130432.000000,89572.000000,86102.000000,143693.000000,141851.00000
mean,12.194034,23.221348,2.360918,5.468232,7.611178,40.035230,14.043426,18.662657,68.880831,51.539116,1017.64994,1015.255889,4.447461,4.509930,16.990631,21.68339
std,6.398495,7.119049,8.478060,4.193704,3.785483,13.607062,8.915375,8.809800,19.029164,20.795902,7.10653,7.037414,2.887159,2.720357,6.488753,6.93665
min,-8.500000,-4.800000,0.000000,0.000000,0.000000,6.000000,0.000000,0.000000,0.000000,0.000000,980.50000,977.100000,0.000000,0.000000,-7.200000,-5.40000
25%,7.600000,17.900000,0.000000,2.600000,4.800000,31.000000,7.000000,13.000000,57.000000,37.000000,1012.90000,1010.400000,1.000000,2.000000,12.300000,16.60000
50%,12.000000,22.600000,0.000000,4.800000,8.400000,39.000000,13.000000,19.000000,70.000000,52.000000,1017.60000,1015.200000,5.000000,5.000000,16.700000,21.10000
75%,16.900000,28.200000,0.800000,7.400000,10.600000,48.000000,19.000000,24.000000,83.000000,66.000000,1022.40000,1020.000000,7.000000,7.000000,21.600000,26.40000
max,33.900000,48.100000,371.000000,145.000000,14.500000,135.000000,130.000000,87.000000,100.000000,100.000000,1041.00000,1039.600000,9.000000,9.000000,40.200000,46.70000


In [131]:
from sklearn.preprocessing import MinMaxScaler

In [132]:
scaler = MinMaxScaler()
scaler.fit(train_inputs[numeric_cols] )

,feature_range,"(0, ...)"
,copy,True
,clip,False


In [133]:
print("Minimum values:", scaler.data_min_)
print("Maximum values:", scaler.data_max_)

Minimum values: [ -8.5  -4.1   0.    0.    0.    6.    0.    0.    0.    0.  980.5 979.
   0.    0.   -5.9  -5.1]
Maximum values: [  33.9   48.1  371.    82.4   14.3  135.    87.    87.   100.   100.
 1041.  1039.6    9.     9.    40.2   46.1]


In [134]:
train_inputs[numeric_cols] = scaler.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = scaler.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])

In [135]:
train_inputs[numeric_cols].describe()

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm
count,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000,101018.000000
mean,0.483307,0.518799,0.006419,0.064103,0.531046,0.265627,0.162390,0.215837,0.686942,0.515524,0.611911,0.596436,0.479635,0.491284,0.492677,0.519651
std,0.148900,0.133632,0.022729,0.037708,0.202591,0.102339,0.102797,0.101352,0.188315,0.205846,0.111137,0.109770,0.253426,0.236163,0.137926,0.132807
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.377358,0.421456,0.000000,0.046117,0.524476,0.193798,0.080460,0.149425,0.570000,0.370000,0.543802,0.528053,0.333333,0.333333,0.392625,0.423828
50%,0.478774,0.507663,0.000000,0.064103,0.531046,0.255814,0.149425,0.218391,0.690000,0.520000,0.611911,0.596436,0.479635,0.491284,0.488069,0.509766
75%,0.589623,0.611111,0.002695,0.065534,0.643357,0.310078,0.218391,0.275862,0.830000,0.650000,0.680992,0.665017,0.666667,0.666667,0.587852,0.609375
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [136]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoder.fit(train_inputs[categorical_cols])
train_inputs_encoded = encoder.transform(train_inputs[categorical_cols])
val_inputs_encoded = encoder.transform(val_inputs[categorical_cols])
test_inputs_encoded = encoder.transform(test_inputs[categorical_cols])


In [138]:
train_inputs_encoded

array([[0., 0., 1., ..., 0., 1., 0.],
       [0., 0., 1., ..., 1., 1., 0.],
       [0., 0., 1., ..., 1., 1., 0.],
       ...,
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 1., 0.]], shape=(101018, 99))